In [1]:
import requests
import time
import platform
import psutil
import subprocess
import os
from statistics import mean, stdev

# =========================================================
# IMPORTANT:
# To force CPU-only inference, start Ollama like this:
#
#   macOS/Linux:
#   OLLAMA_NO_GPU=1 ollama serve
#
#
# Then run this notebook cell.
# =========================================================

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "ReqBrain"
N_RUNS = 5
WARMUP_RUNS = 1

PROMPT = """Write five software requirements for a login system.
Each requirement should be clear, concise, and testable."""

OPTIONS = {
    "temperature": 0.0,
}

def check_ollama_server():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=10)
        r.raise_for_status()
        return True
    except Exception as e:
        print("Could not connect to Ollama server:", e)
        print("Make sure Ollama is running.")
        return False

def get_ollama_version():
    try:
        out = subprocess.check_output(["ollama", "--version"], text=True).strip()
        return out
    except Exception:
        return "Unavailable"

def run_once():
    payload = {
        "model": MODEL_NAME,
        "prompt": PROMPT,
        "stream": False,
        "options": OPTIONS
    }

    start = time.time()
    response = requests.post(OLLAMA_URL, json=payload, timeout=300)
    end = time.time()
    response.raise_for_status()
    data = response.json()

    eval_count = data.get("eval_count")
    eval_duration = data.get("eval_duration")  # ns
    wall_time = end - start

    tps = None
    ms_per_token = None
    if eval_count is not None and eval_duration is not None and eval_count > 0 and eval_duration > 0:
        tps = eval_count / (eval_duration / 1e9)
        ms_per_token = (eval_duration / 1e6) / eval_count

    return {
        "tokens": eval_count,
        "tps": tps,
        "ms_per_token": ms_per_token,
        "wall_time": wall_time
    }

if not check_ollama_server():
    raise SystemExit

print("===== SYSTEM INFORMATION =====")
print("Platform:", platform.platform())
print("Processor:", platform.processor())
print("Machine:", platform.machine())
print("CPU cores (logical):", psutil.cpu_count(logical=True))
print("CPU cores (physical):", psutil.cpu_count(logical=False))
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"RAM: {ram_gb:.2f} GB")
print("Ollama version:", get_ollama_version())

print("\n===== CPU-ONLY EXECUTION NOTE =====")
print("This script cannot force CPU mode by itself.")
print("CPU-only execution depends on how Ollama was started.")
print("If you started Ollama with OLLAMA_NO_GPU=1, this run is CPU-only.")

print("\n===== MODEL CONFIGURATION =====")
print("Model:", MODEL_NAME)
print("Runs:", N_RUNS)
print("Prompt length:", len(PROMPT.split()), "words")
print("Options:", OPTIONS)

print("\nRunning warmup...")
for _ in range(WARMUP_RUNS):
    _ = run_once()

print("\n===== RUNNING BENCHMARK =====")
results = []

for i in range(N_RUNS):
    r = run_once()
    results.append(r)

    tps_str = f"{r['tps']:.2f}" if r["tps"] is not None else "NA"
    ms_str = f"{r['ms_per_token']:.2f}" if r["ms_per_token"] is not None else "NA"

    print(
        f"Run {i+1}: tokens={r['tokens']}, "
        f"tps={tps_str}, "
        f"ms/token={ms_str}, "
        f"wall={r['wall_time']:.2f}s"
    )

valid_tps = [r["tps"] for r in results if r["tps"] is not None]
valid_ms = [r["ms_per_token"] for r in results if r["ms_per_token"] is not None]

print("\n===== SUMMARY =====")
if len(valid_tps) > 0:
    tps_avg = mean(valid_tps)
    tps_std = stdev(valid_tps) if len(valid_tps) > 1 else 0.0
    print(f"Avg tokens/sec: {tps_avg:.2f} (std {tps_std:.2f})")
else:
    tps_avg = None
    print("Avg tokens/sec: NA")

if len(valid_ms) > 0:
    ms_avg = mean(valid_ms)
    ms_std = stdev(valid_ms) if len(valid_ms) > 1 else 0.0
    print(f"Avg ms/token: {ms_avg:.2f} (std {ms_std:.2f})")
else:
    ms_avg = None
    print("Avg ms/token: NA")

print("\n===== PAPER-READY SENTENCE =====")
if tps_avg is not None and ms_avg is not None:
    print(
        f"In our local deployment setup "
        f"({psutil.cpu_count(logical=False)} CPU cores, {ram_gb:.0f} GB RAM), "
        f"inference with {MODEL_NAME} achieved {tps_avg:.2f} tokens/s "
        f"(≈{ms_avg:.2f} ms/token) on average over {len(valid_tps)} runs."
    )
else:
    print("Could not compute final metrics.")

===== SYSTEM INFORMATION =====
Platform: Linux-6.8.0-106-generic-x86_64-with-glibc2.35
Processor: x86_64
Machine: x86_64
CPU cores (logical): 32
CPU cores (physical): 16
RAM: 125.64 GB
Ollama version: ollama version is 0.21.1

===== CPU-ONLY EXECUTION NOTE =====
This script cannot force CPU mode by itself.
CPU-only execution depends on how Ollama was started.
If you started Ollama with OLLAMA_NO_GPU=1, this run is CPU-only.

===== MODEL CONFIGURATION =====
Model: ReqBrain
Runs: 5
Prompt length: 16 words
Options: {'temperature': 0.0}

Running warmup...

===== RUNNING BENCHMARK =====
Run 1: tokens=199, tps=10.26, ms/token=97.49, wall=19.60s
Run 2: tokens=199, tps=10.26, ms/token=97.51, wall=19.60s
Run 3: tokens=199, tps=10.26, ms/token=97.44, wall=19.58s
Run 4: tokens=199, tps=10.26, ms/token=97.48, wall=19.59s
Run 5: tokens=199, tps=10.25, ms/token=97.51, wall=19.60s

===== SUMMARY =====
Avg tokens/sec: 10.26 (std 0.00)
Avg ms/token: 97.49 (std 0.03)

===== PAPER-READY SENTENCE =====
In

In [2]:
# Source - https://stackoverflow.com/a/13078519
# Posted by dbn, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-25, License - CC BY-SA 4.0

import os, platform, subprocess, re

def get_processor_name():
    if platform.system() == "Windows":
        return platform.processor()
    elif platform.system() == "Darwin":
        os.environ['PATH'] = os.environ['PATH'] + os.pathsep + '/usr/sbin'
        command ="sysctl -n machdep.cpu.brand_string"
        return subprocess.check_output(command).strip()
    elif platform.system() == "Linux":
        command = "cat /proc/cpuinfo"
        all_info = subprocess.check_output(command, shell=True).decode().strip()
        for line in all_info.split("\n"):
            if "model name" in line:
                return re.sub( ".*model name.*:", "", line,1)
    return ""

In [3]:
get_processor_name()

' AMD Ryzen Threadripper PRO 3955WX 16-Cores'